In [1]:
# Test PyDeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import random
adata_sub = adata[adata.obs['cell_type_final']=='SCH Gaba']
random.seed(2115)
adata.X.max()
adata
pbs = [] ### With replicates 

samples_Deseq = ['3159-1','2670-1','3160-1','3160-2','2505-1','2505-2']

for sample in samples_Deseq:
    samp_adata_sub = adata_sub[adata_sub.obs['sample']==sample]
    samp_adata_sub.X = samp_adata_sub.layers['counts']

    random.seed(2115)
    
    indices = list(samp_adata_sub.obs_names)
    random.shuffle(indices)
    indices = np.array_split(np.array(indices), 3)

    for i, pseudo_rep in enumerate(indices):

        rep_adata = sc.AnnData(X = samp_adata_sub[indices[i]].X.sum(axis=0),
                            var = samp_adata_sub[indices[i]].var[[]])
        
        rep_adata.obs_names = [sample + '_' + str(i)]
        rep_adata.obs['Genotype'] = samp_adata_sub.obs['Genotype'].iloc[0]
        rep_adata.obs['replicate'] = i

        pbs.append(rep_adata)
pbs = [] ### No pseudoreplicates
for sample in samples_Deseq:
    samp_adata_sub = adata_sub[adata_sub.obs['sample']==sample]
    samp_adata_sub.X = samp_adata_sub.layers['counts']


    rep_adata = sc.AnnData(X = samp_adata_sub.X.sum(axis=0),
                        var = samp_adata_sub.var[[]])
    
    rep_adata.obs_names = [sample]
    rep_adata.obs['Genotype'] = samp_adata_sub.obs['Genotype'].iloc[0]
    rep_adata.obs['ZT'] = samp_adata_sub.obs['ZT'].iloc[0]

    pbs.append(rep_adata)
pb = sc.concat(pbs)
# pb.obs
counts = pd.DataFrame(pb.X, columns = pb.var_names)
# counts
counts
# dds = DeseqDataSet(counts = counts,
#                    metadata = pb.obs,
#                    design_factors = ['Genotype',"replicate"]) ### With replicates

dds = DeseqDataSet(counts = counts,
                   metadata = pb.obs,
                   design_factors = ['Genotype'])

# dds = DeseqDataSet(counts = counts,
#                    metadata = pb.obs,
#                    design_factors = ['Genotype',"ZT"])
dds
sc.pp.filter_genes(dds, min_cells = 1)
dds
dds.deseq2()
stat_res = DeseqStats(dds, n_cpus = 8, contrast = ('Genotype','APP','WT'))

stat_res.summary()
de = stat_res.results_df
de.sort_values('padj',ascending=True).head(5)
de_filter = de[(abs(de['log2FoldChange'])>0.26) & (de['padj']<0.05)]
print(len(de_filter))
de_filter[de_filter['log2FoldChange'] <0 ].index
sc.tl.pca(dds)
sc.pl.pca(dds, color='Genotype',size = 200)
plt.figure(figsize=(5,5))
# plt.vlines(x=(-0.26,0.26), ymin=df_filter[key]['pvals_adj'].min(), ymax=1, color = "black", linestyles='dashed')
# plt.hlines(y=0.05, xmin=-1, xmax=1, color = "black", linestyles='dashed')

plt.scatter(x=de['log2FoldChange'], y = de['padj'], s=2, alpha= 0.75, color = "grey", edgecolors=None)
plt.scatter(x= de_filter['log2FoldChange'], y=de_filter['padj'], s=5, alpha=1, color = 'red')
for idx, gene in enumerate(de_filter.index):
    plt.text(x=de_filter['log2FoldChange'][gene], y=de_filter['padj'][gene], s=str(gene),
              color = 'black', fontsize = 8, ha= 'center')
plt.yscale('log')
# plt.xlim(-2,2)
# plt.ylim(0,100000)
plt.gca().invert_yaxis()
plt.xlabel('Log2 Foldchange')
plt.ylabel('Adjusted p-value')
# plt.title(f"DEG: {key} - Whole Brain")
# plt.savefig(f'Gallery/{today}/volcano_plot_{key}.svg')

## DeSeq2 loop
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import random
import pandas as pd
import scanpy as sc
import numpy as np
dir_notebook = '../notebook'
name_dir = 'circa-SD'
adata = sc.read_h5ad(f'{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz')

# adata = adata[~(adata.obs['cell_type_final']=="Undefined")]
# adata = adata[~(adata.obs['cell_type_final']=="AD Glut")]


list_celltypes = adata.obs['cell_type_final'].unique()
ddf = []
ddf_filter = []
count = 0
condition = 'run'
groups = adata.obs[condition].unique()

for cell in list_celltypes:  ### With replicates 
    count+= 1
    print(cell, count, "/", len(list_celltypes))
    adata_sub = adata[adata.obs['cell_type_final']==cell]

    
    pbs = [] 
    for sample in adata_sub.obs['sample'].unique():
        
        samp_adata_sub = adata_sub[adata_sub.obs['sample']==sample]
        
        samp_adata_sub.X = samp_adata_sub.layers['counts']

        random.seed(20150201)

        indices = list(samp_adata_sub.obs_names)
        random.shuffle(indices)
        indices = np.array_split(np.array(indices), 3)

        for i, pseudo_rep in enumerate(indices):

            rep_adata = sc.AnnData(X = samp_adata_sub[indices[i]].X.sum(axis=0),
                                var = samp_adata_sub[indices[i]].var[[]])
            
            rep_adata.obs_names = [sample + '_' + str(i)]
            rep_adata.obs[condition] = samp_adata_sub.obs[condition].iloc[0]
            rep_adata.obs['replicate'] = i

            pbs.append(rep_adata)
    
    pb = sc.concat(pbs)
    counts = pd.DataFrame(pb.X, columns = pb.var_names)
    dds = DeseqDataSet(counts = counts,
                   metadata = pb.obs,
                   design_factors = [condition])
    sc.pp.filter_genes(dds, min_cells = 1)
    dds.deseq2()
    stat_res = DeseqStats(dds, n_cpus = 32, contrast = (condition,groups[1],groups[0]))

    stat_res.summary() 
    de = stat_res.results_df
    de_filter = de[(abs(de['log2FoldChange'])>0.26) & (de['padj']<0.05)]

    ddf.append(de)
    ddf_filter.append(de_filter)


import os
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/foldchanges_DeSeq2/celltype"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/foldchanges_DeSeq2/celltype")

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges_DeSeq2/celltype/DEG_DeSeq2_celltype_no-filter.xlsx', engine='xlsxwriter')
for j in range(len(list_celltypes)):
    ddf[j].to_excel(writer, sheet_name=list_celltypes[j], index=True)
writer.close()

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges_DeSeq2/celltype/DEG_DeSeq2_celltype_filter.xlsx', engine='xlsxwriter')
for j in range(len(list_celltypes)):
    ddf_filter[j].to_excel(writer, sheet_name=list_celltypes[j], index=True)
writer.close()
for key in range(len(ddf_filter)):
    print(list_celltypes[key],f"({key}) : ", len(ddf_filter[key]))
ddf[0].head(1)
df_deg_quant = []
# expressed = [len(df_expressed[key]) for key in df_expressed.keys()]
deg_quant = [len(ddf_filter[key]) for key in range(len(ddf_filter))]
deg_up = [len(ddf_filter[key][ddf_filter[key]['log2FoldChange'] > 0]) for key in range(len(ddf_filter))]
deg_down = [len(ddf_filter[key][ddf_filter[key]['log2FoldChange'] < 0]) for key in range(len(ddf_filter))]

df_deg_quant = pd.DataFrame(data = {'Celltype' : list_celltypes,
                                    # 'Expressed' : expressed,
                                    'nb_DEG' : deg_quant,
                                    'Upregulated': deg_up,
                                    'Downregulated': deg_down
                                    })

df_deg_quant.index = df_deg_quant['Celltype']
df_deg_quant.sort_values(by='Downregulated')


ModuleNotFoundError: No module named 'pydeseq2'